# Introduction

We are now moving to the final part of the workshop, which involves formulating business recommendations. Our tasks are:
- Determining a global betting odds,
- Dividing the dataset into categories: A, B, C, D, where A is the best group and D is the weakest group,
- Determining the risk of odds based on accepted parameters for each category.

As the last task, in a discussion format, we must consider the fact that we are a new betting company. When formulating our recommendations, we need to identify the risks that may affect our operations. We will perform this task together in a brainstorming session.

# Notebook Configuration

## Import necessary libraries

In [7]:
import os
import pandas as pd
# by default it uses "/notebooks" as working directory, so I changed it to the root of the project:
os.chdir(r"/workspaces/CLab_workshop/") 
print(os.getcwd())

/workspaces/CLab_workshop


## Loading data into the workspace

> Remember to correctly specify the column separator

In [8]:
df = pd.read_csv(r"data/processed/hockey_teams.csv", sep = ";", encoding = "utf-8", decimal = ".")

### Checking data loading accuracy

In [9]:
df

,team,scored_goals,received_goals,goals_ratio,victory_percentage
0,Anaheim Ducks,1389,1354,35,0.522333
1,Atlanta Thrashers,2465,3014,-549,0.379182
2,Boston Bruins,5074,4823,251,0.484905
3,Buffalo Sabres,5076,4694,382,0.475143
4,Calgary Flames,4920,4896,24,0.453476
5,Carolina Hurricanes,3104,3254,-150,0.448500
6,Chicago Blackhawks,4875,4758,117,0.454286
7,Colorado Avalanche,3939,3534,405,0.516062
8,Columbus Blue Jackets,2220,2744,-524,0.379182
9,Dallas Stars,4123,3729,394,0.516889


# Determining Betting Odds

Let's review the content of the page: [click](https://trustbet.pl/kursy-bukmacherskie/), where information about methods for determining betting odds can be found. First, we will determine a global odd, which will be the starting point for our analysis (the so-called _baseline scenario_). At this point, we ignore the margin and assume that we are calculating the decimal odd.

Here is the list of steps to be performed to obtain the desired value:
- we will complete the definition of the `get_betting_odds` function, which will take `probability` of a given event as a parameter. We will use it multiple times, so it is worth preparing its implementation now
- then we need to appropriately aggregate the set and determine the **global** probability of the team's victory.

## Implementations of the `get_betting_odds` function

In [10]:
def get_betting_odds(probability):
    if probability == 0:
        raise ZeroDivisionError
    else:
        return 1/probability

### Some tests to check the correctness of the implementation

In [11]:
def test_get_betting_odds():
    assert get_betting_odds(1) == 1, "Expected 1"
    assert get_betting_odds(0.5) == 2, "Expected 2"
    assert get_betting_odds(0.25) == 4, "Expected 4"
    assert get_betting_odds(0.1) == 10, "Expected 10"
    try:
        get_betting_odds(0)
    except ZeroDivisionError:
        pass
    else:
        assert False, "Expected ZeroDivisionError"

    print("All tests passed!")

test_get_betting_odds()

All tests passed!


### Determining the global odds

Here, determine the probability of any team winning

In [ ]:
# I hope this task is about loading win percentages from the dataframe.
# It doesn't help that linked instructions are in Polish and I am relying on Google translate.
# of course this crude method would not work for real world betting company.

df_win_probability = pd.DataFrame(df[["team","victory_percentage"]])
print(df_win_probability["victory_percentage"].dtype)
df_win_probability

float64


,team,victory_percentage
0,Anaheim Ducks,0.522333
1,Atlanta Thrashers,0.379182
2,Boston Bruins,0.484905
3,Buffalo Sabres,0.475143
4,Calgary Flames,0.453476
5,Carolina Hurricanes,0.448500
6,Chicago Blackhawks,0.454286
7,Colorado Avalanche,0.516062
8,Columbus Blue Jackets,0.379182
9,Dallas Stars,0.516889


Set the global rate here using the `get_betting_odds` function. Round the result to two decimal places.

In [21]:
odds = []

df_win_probability["victory_percentage"].apply(lambda victory_perc : odds.append(round(get_betting_odds(victory_perc),2)))

series_odds = pd.Series(odds)

df_win_probability["odds"] = series_odds

df_sorted_by_odds = df_win_probability.sort_values(by="odds").reset_index(drop=True)

df_sorted_by_odds

,team,victory_percentage,odds
0,Detroit Red Wings,0.586000,1.71
1,New Jersey Devils,0.534333,1.87
2,Anaheim Ducks,0.522333,1.91
3,Dallas Stars,0.516889,1.93
4,Colorado Avalanche,0.516062,1.94
5,Pittsburgh Penguins,0.498810,2.00
6,Philadelphia Flyers,0.496952,2.01
7,Boston Bruins,0.484905,2.06
8,St. Louis Blues,0.482571,2.07
9,Vancouver Canucks,0.480571,2.08


# Team Categorization

Let's discuss how we can classify teams into _leagues_. We want to establish 4 leagues:
- A - league consisting of the best teams,
- B - league consisting of good teams,
- C - league consisting of average teams,
- D - league consisting of the weakest teams.

The above terms are quite subjective, so for the purpose of this exercise, we will adopt the following assumptions:
- A - the top 5% of teams,
- B - teams performing better than 70% of the group but worse than league A,
- C - teams performing better than 20% of the group but worse than league B,
- D - the remaining teams.

To accomplish this task, we will additionally implement the function `assign_team_to_league`.

> Note: This task looks unassuming, but it is difficult. Remember that during the class, you have access to the instructor, and later to a mentor.

## Determination of cutoff points for individual leagues

In [ ]:
"""
here I first wrote code in subsequent cell, which is anchored in my dataset, and then I abstracted it
into this function:
"""
def assign_team_to_league(x):
    pass

In [36]:

no_of_teams = len(df_sorted_by_odds)

print(no_of_teams) # still 35

no_in_A_league = no_of_teams*0.05

print(no_in_A_league) 
"""
there just 1.75 teams in 'A league', which would be unworkable for
real sports league, but that is to be expected since
NHL as of now has just 32 teams(https://en.wikipedia.org/wiki/National_Hockey_League).
But since the assumed purpose of the exercise is not to establish league but to
determine odds of winning the match, imho it is ok.
Imho authors should call it brackets rather than leagues, to prevent confusion.
"""
A_league = df_sorted_by_odds.loc[0:1] # I am rounding 1.75 to 2
print(f"A league:\n{A_league}")

no_in_B_league = no_of_teams - no_of_teams*0.7 - len(A_league)

print(no_in_B_league) # 8.5, which I am rounding up to 9

B_league = df_sorted_by_odds.loc[2:10]
print(f"B league:\n{B_league}")

no_in_C_league = no_of_teams - no_of_teams*0.2 - len(A_league) - len(B_league)
print(no_in_C_league) # exactly 17

C_league = df_sorted_by_odds.loc[11:27]
print(len(C_league)) # math control :-))
print(f"C league:\n{C_league}")

no_in_D_league = no_of_teams - len(A_league) - len(B_league) - len(C_league)
print(no_in_D_league) # 7 which checks out with C league finishing with [27], since last team is [34]

D_league = df_sorted_by_odds.loc[28:34]
print(f"D league:\n{D_league}")


35
1.75
A league:
                team  victory_percentage  odds
0  Detroit Red Wings            0.586000  1.71
1  New Jersey Devils            0.534333  1.87
8.5
B league:
                   team  victory_percentage  odds
2         Anaheim Ducks            0.522333  1.91
3          Dallas Stars            0.516889  1.93
4    Colorado Avalanche            0.516062  1.94
5   Pittsburgh Penguins            0.498810  2.00
6   Philadelphia Flyers            0.496952  2.01
7         Boston Bruins            0.484905  2.06
8       St. Louis Blues            0.482571  2.07
9     Vancouver Canucks            0.480571  2.08
10  Washington Capitals            0.477143  2.10
17.0
17
C league:
                       team  victory_percentage  odds
11           Buffalo Sabres            0.475143  2.10
12      Nashville Predators            0.471769  2.12
13         New York Rangers            0.468952  2.13
14       Montreal Canadiens            0.461952  2.16
15      Toronto Maple Leafs            

## Determination of odds per league

Here we set the betting odds for each league, which will allow us to draw final conclusions and establish the basic odds for individual teams.

> Remember: After generating the results, it is worth checking if they are reasonable.

# Discussion

We have obtained certain odds values for each league. But how does this translate into real business? The entire task was about determining certain values from which a bookmaker can begin operations. Correct determination of these values is critical to attract customers to place bets with us, and on the other hand, inappropriate determination may lead to financial losses in the first days of operation.

For this reason, before translating the results and recommendations into business objectives, the analysis is subjected to discussion. Therefore, we will now take on a review role and would like to verify the steps. To that end, we will collectively discuss and critique our work by answering the following questions together:
- What elements of the analysis were simplified? What was omitted in the analysis?
- Are there any inconsistencies in the estimated odds? What are they?
- How can we improve the odds estimates?
- How can we enrich our initial dataset to make the estimates more accurate and less risky?
- How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?

This is a discussion panel, and every idea is valuable here.
